In [ ]:
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from dotenv import 

In [4]:
## Creating a Template
prompt = PromptTemplate(template = "give me answer of this {question} in 50 words", input_variables =['question'])

## creating the model 
model = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

chain = prompt | model 
output = chain.invoke({'question','What is chain in Langchain'})
print(output.content)

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
## Task 6:  Conditional Chain


In [ ]:

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
import re

## Creating Documents
documents = [
    Document(
        page_content="LangChain is a framework for developing applications using language models.",
        metadata={"source": "LangChain documentation"}
    ),
    Document(
        page_content="Prompt templates help create prompts using dynamic input variables.",
        metadata={"source": "LangChain documentation"}
    ),
    Document(
        page_content="Retrievers return documents relevant to a user's query.",
        metadata={"source": "LangChain documentation"}
    )
]

## Defining Model
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

## Creating Prompt Templates
reterival_prompt = PromptTemplate(
    template="Answer using this information: {context}\nQuestion: {question}",
    input_variables=["context", "question"]
)

direct_prompt = PromptTemplate(
    template="Answer this question in 50 words.\nQuestion: {question}",
    input_variables=["question"]
)

## Creating Retrieval Function
def reterival_context(question):

    question_words = set(re.findall(r"\b\w+\b", question.lower()))

    stopwords = {"what", "is", "a", "an", "the", "who", "how", "when", "where", "which", "using", "for", "of", "to"}

    question_words -= stopwords

    for document in documents:
        document_words = set(
            re.findall(r"\b\w+\b", document.page_content.lower())
        )

        if question_words & document_words:
            return document.page_content

    return "No relevant information found"


## Creating Conditional Logic
def answer_question(question):

    context = reterival_context(question)

    if context != "No relevant information found":

        chain = reterival_prompt | model

        output = chain.invoke({
            "context": context,
            "question": question
        })

        print("Method: Retrieval")

    else:

        chain = direct_prompt | model

        output = chain.invoke({
            "question": question
        })

        print("Method: Direct LLM")

        print("Answer:", output.content)
        return output.content


## Testing
answer_question("What is LangChain?")
answer_question("What is Python?")
answer_question("Write a poem about nature.")


Method: Retrieval
Method: Direct LLM
Answer: Python is a high-level, interpreted programming language known for its readability and versatility. Created by Guido van Rossum in 1991, it supports multiple paradigms, including procedural, object‑oriented, and functional styles. Its extensive standard library and vibrant community enable rapid development across web, data science, automation, and more in various domains.
Method: Direct LLM
Answer: Morning mist drapes the silent forest,  
Sunlight pierces emerald canopies,  
River hums a crystal lullaby,  
Mountains stand as ancient guardians,  
Wildflowers whisper colors to the wind,  
Birds sketch melodies across azure skies,  
Nature breathes, endless, pure, unbound.  
Evening shadows dance, and stars awaken the dreaming earth softly in quiet reverie.


'Morning mist drapes the silent forest,  \nSunlight pierces emerald canopies,  \nRiver hums a crystal lullaby,  \nMountains stand as ancient guardians,  \nWildflowers whisper colors to the wind,  \nBirds sketch melodies across azure skies,  \nNature breathes, endless, pure, unbound.  \nEvening shadows dance, and stars awaken the dreaming earth softly in quiet reverie.'

In [ ]:
## Task 7: Parallel Chain


In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
import re


model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
) 

parser = StrOutputParser()


## PromptTemplate

## Answer prompt Template
answer_prompt = PromptTemplate(
                template = 'Answer this question im max 1000 words: {question}', 
                input_variables = ['question'])

## Summary Prompt
summary_prompt = PromptTemplate(
                template = 'Give a short summary about this in 100 words: {question}', 
                input_variables = ['question'])  

## FollowUp Prompt
followup_prompt = PromptTemplate(
                template = 'Genearate 3 follow up question about: {question}', 
                input_variables = ['question'])  

 ## Creating Individual Chains for Each
answer_chain =   answer_prompt | model | parser
summary_chain = summary_prompt | model | parser
followup_chain = followup_prompt | model | parser


 ## Parallel Chain
parallel_chain = RunnableParallel({
    "answer": answer_chain,
    "summary": summary_chain,
    "follow_up_questions": followup_chain 
    })


  ## Invoke and combine outputs
output = parallel_chain.invoke({"question": "What is Langchain? "})

# print(output['answer'])
# print(output['summary'])


print("Answer:", output["answer"])
print("Summary:", output["summary"])
print("Follow-up Questions:", output["follow_up_questions"])


                              


LangChain is an open‑source framework that simplifies building applications powered by large language models (LLMs). It provides modular components—prompts, models, memory, agents, and data connectors—allowing developers to chain together LLM calls with external tools, APIs, and databases. By abstracting prompt management, context handling, and output parsing, LangChain accelerates creation of chatbots, question‑answering systems, and autonomous agents. It supports multiple LLM providers (OpenAI, Anthropic, Cohere, etc.) and integrates with vector stores, document loaders, and retrieval‑augmented generation pipelines. The library’s extensibility and community‑driven ecosystem make it a popular choice for rapid prototyping and production‑grade AI applications across diverse industries and research domains.
